# STEP 11 - find the real mDeBERTa bug, and train BanglaT5 properly

**mDeBERTa.** The log shows a healthy forward pass (loss 0.79) and failure on the first
step *after* the first optimizer update, at both micro-batch 8 and 4. At an initial
learning rate near 2e-7 with clipping, weights can only explode if the **gradients** are
already non-finite - and `clip_grad_norm_` does not sanitise NaN, so one NaN gradient
turns the norm into NaN and every parameter with it.

Our loop differs from a standard HuggingFace fine-tune in four ways. Part A tests them one
at a time on 12 steps each and reports **which parameter's gradient goes bad first**, so we
fix the cause rather than guess:

| | fixed padding | decay on LayerNorm | grad checkpointing |
|---|---|---|---|
| A as-is | yes | yes | yes |
| B dynamic padding | **no** | yes | yes |
| C no decay on LayerNorm/bias | yes | **no** | yes |
| D no checkpointing | yes | yes | **no** |
| E all three | no | no | no |

Part B then trains mDeBERTa with whatever configuration survives, adding a NaN-gradient
skip so a single bad batch cannot kill a 40-minute run.

**BanglaT5.** It trained in Step 10 but did not learn: the loss sat at ~0.6 for both
epochs and clean accuracy was 0.768 against BanglaBERT's 0.922. T5 needs a far higher
learning rate than BERT-family models. Part C retrains it at 1e-4, which is standard for
T5 fine-tuning, so the same-lab comparison uses a model that actually works.

**Settings:** GPU **T4 x2**, Internet **ON**, data **`naurosromim/bdshs`**. About 2.5 hours.


In [ ]:
!pip install -q "emoji==1.7.0" regex ftfy uharfbuzz sentencepiece protobuf
!pip install -q git+https://github.com/csebuetnlp/normalizer

In [ ]:
%%writefile /kaggle/working/render_attack.py
"""
Rendering-verified zero-width attacks and defenses for Bangla.

Threat model
------------
An insertion counts as imperceptible only if HarfBuzz shapes the word to the
same glyph ids and glyph positions, in every reference font, with and without
it. HarfBuzz is the shaper used by Chrome, Android and Firefox.

The earlier rule-based attacks (strict / semi) are kept for comparison, but
every number in the paper should come from the render-gated functions here.
"""
import os
import re
import random
import urllib.request
import unicodedata
from functools import lru_cache

import uharfbuzz as hb

ZWNJ, ZWJ, ZWSP = "‌", "‍", "​"
HASANT = "্"
ZW = (ZWNJ, ZWJ, ZWSP)

# Open-licensed Bangla fonts, all downloadable on Kaggle.
FONT_URLS = {
    "NotoSansBengali":  "https://github.com/notofonts/notofonts.github.io/raw/main/fonts/NotoSansBengali/unhinted/ttf/NotoSansBengali-Regular.ttf",
    "NotoSerifBengali": "https://github.com/notofonts/notofonts.github.io/raw/main/fonts/NotoSerifBengali/unhinted/ttf/NotoSerifBengali-Regular.ttf",
    "HindSiliguri":     "https://raw.githubusercontent.com/google/fonts/main/ofl/hindsiliguri/HindSiliguri-Regular.ttf",
    "TiroBangla":       "https://raw.githubusercontent.com/google/fonts/main/ofl/tirobangla/TiroBangla-Regular.ttf",
    "BalooDa2":         "https://raw.githubusercontent.com/google/fonts/main/ofl/balooda2/BalooDa2%5Bwght%5D.ttf",
    "AnekBangla":       "https://raw.githubusercontent.com/google/fonts/main/ofl/anekbangla/AnekBangla%5Bwdth,wght%5D.ttf",
    "Galada":           "https://raw.githubusercontent.com/google/fonts/main/ofl/galada/Galada-Regular.ttf",
    "Mina":             "https://raw.githubusercontent.com/google/fonts/main/ofl/mina/Mina-Regular.ttf",
}


def fetch_fonts(dest, names=None):
    os.makedirs(dest, exist_ok=True)
    paths = {}
    for name in names or FONT_URLS:
        p = os.path.join(dest, name + ".ttf")
        if not os.path.exists(p):
            urllib.request.urlretrieve(FONT_URLS[name], p)
        paths[name] = p
    return paths


def is_bangla(ch):
    """True for one Bangla code point.

    An empty string returns False, so callers may pass a missing neighbour
    (start or end of string) without guarding first."""
    return len(ch) == 1 and 0x0980 <= ord(ch) <= 0x09FF


# ------------------------------------------------------------------ slots

def inert_slots(word):
    """Between two Bangla chars, neither of them hasant (the old `semi` set)."""
    return [i for i in range(1, len(word))
            if is_bangla(word[i - 1]) and is_bangla(word[i])
            and HASANT not in (word[i - 1], word[i])]


def strict_slots(word):
    """Both neighbours Bangla base letters (the old `strict` set)."""
    return [i for i in inert_slots(word)
            if unicodedata.category(word[i - 1]) == "Lo"
            and unicodedata.category(word[i]) == "Lo"]


def hasant_slots(word):
    """Immediately before or after a hasant. Position-aware defense keeps these."""
    out = set()
    for h, c in enumerate(word):
        if c != HASANT:
            continue
        if h > 0 and is_bangla(word[h - 1]):
            out.add(h)
        if h + 1 < len(word) and is_bangla(word[h + 1]):
            out.add(h + 1)
    return sorted(out)


SLOT_MODES = {
    "inert":  inert_slots,
    "strict": strict_slots,
    "hasant": hasant_slots,
    "both":   lambda w: inert_slots(w) + hasant_slots(w),
}


def _build(word, chosen, char):
    return "".join((char if i in chosen else "") + c for i, c in enumerate(word))


def _parts(text):
    return re.split(r"(\s+)", text)


# --------------------------------------------------------------- renderer

class Renderer:
    def __init__(self, font_paths):
        self.fonts = {n: hb.Font(hb.Face(hb.Blob.from_file_path(p), 0))
                      for n, p in font_paths.items()}
        self.names = list(self.fonts)
        self._shape = lru_cache(maxsize=2_000_000)(self._shape_uncached)

    def _shape_uncached(self, name, text):
        buf = hb.Buffer()
        buf.add_str(text)
        buf.guess_segment_properties()
        buf.flags = hb.BufferFlags.REMOVE_DEFAULT_IGNORABLES
        hb.shape(self.fonts[name], buf, {})
        return (tuple(g.codepoint for g in buf.glyph_infos),
                tuple((p.x_advance, p.y_advance, p.x_offset, p.y_offset)
                      for p in buf.glyph_positions))

    def same(self, a, b, fonts=None):
        return all(self._shape(n, a) == self._shape(n, b) for n in (fonts or self.names))

    def visible_in(self, a, b):
        """Names of fonts in which a and b render differently."""
        return [n for n in self.names if self._shape(n, a) != self._shape(n, b)]

    # ------------------------------------------------------------ attacks

    def attack(self, text, budget="ALL", char=ZWNJ, seed=0, mode="inert", fonts=None):
        """Insert zero-width chars only where the word still renders identically.

        Candidates are shuffled and accepted one at a time, each checked against
        the clean word with all previously accepted insertions in place.
        Returns (attacked_text, n_inserted)."""
        parts = _parts(text)
        rng = random.Random(seed)
        slots = SLOT_MODES[mode]
        cands = [(pi, i) for pi, p in enumerate(parts)
                 if p and not p.isspace() for i in slots(p)]
        rng.shuffle(cands)
        chosen, n = {}, 0
        for pi, i in cands:
            if budget != "ALL" and n >= budget:
                break
            trial = chosen.get(pi, set()) | {i}
            if self.same(parts[pi], _build(parts[pi], trial, char), fonts):
                chosen[pi] = trial
                n += 1
        out = "".join(_build(p, chosen[pi], char) if pi in chosen else p
                      for pi, p in enumerate(parts))
        return out, n

    # ------------------------------------------------------------ defense

    def defend_render_equivalence(self, text, fonts=None):
        """Delete a zero-width char iff removing it leaves rendering unchanged in
        every reference font. Only chars whose nearest non-zero-width neighbours
        on both sides are Bangla are considered, so emoji ZWJ sequences survive."""
        parts = _parts(text)
        for pi, w in enumerate(parts):
            if not any(z in w for z in ZW):
                continue
            i = 0
            while i < len(w):
                if w[i] in ZW:
                    l = i - 1
                    while l >= 0 and w[l] in ZW:
                        l -= 1
                    r = i + 1
                    while r < len(w) and w[r] in ZW:
                        r += 1
                    if l >= 0 and r < len(w) and is_bangla(w[l]) and is_bangla(w[r]):
                        cand = w[:i] + w[i + 1:]
                        if self.same(w, cand, fonts):
                            w = cand
                            continue
                i += 1
            parts[pi] = w
        return "".join(parts)


# ------------------------------------------------ rule-based (for comparison)

def attack_ungated(text, budget="ALL", char=ZWNJ, seed=0, mode="inert"):
    parts = _parts(text)
    rng = random.Random(seed)
    slots = SLOT_MODES[mode]
    cands = [(pi, i) for pi, p in enumerate(parts) if p and not p.isspace() for i in slots(p)]
    rng.shuffle(cands)
    if budget != "ALL":
        cands = cands[:budget]
    chosen = {}
    for pi, i in cands:
        chosen.setdefault(pi, set()).add(i)
    out = "".join(_build(p, chosen[pi], char) if pi in chosen else p for pi, p in enumerate(parts))
    return out, len(cands)


def defend_strip_all(text):
    for z in ZW:
        text = text.replace(z, "")
    return text


def defend_nfc(text):
    return unicodedata.normalize("NFC", text)


def defend_position_aware(text):
    out = []
    for i, ch in enumerate(text):
        if ch in (ZWNJ, ZWJ):
            prev = text[i - 1] if i > 0 else ""
            nxt = text[i + 1] if i + 1 < len(text) else ""
            if HASANT in (prev, nxt):
                out.append(ch)
            continue
        if ch == ZWSP:
            continue
        out.append(ch)
    return "".join(out)


def zw_stats(texts):
    docs = total = functional = 0
    for t in texts:
        hit = False
        for i, ch in enumerate(t):
            if ch not in (ZWNJ, ZWJ):
                continue
            hit = True
            total += 1
            prev = t[i - 1] if i > 0 else ""
            nxt = t[i + 1] if i + 1 < len(t) else ""
            functional += HASANT in (prev, nxt)
        docs += hit
    return dict(n_docs=len(texts), docs_with_zw=docs, total_zw=total,
                functional=functional, inert=total - functional)


In [ ]:
import sys, os, glob, time, random, warnings
warnings.filterwarnings("ignore")
sys.path.insert(0, "/kaggle/working")
import importlib, render_attack as ra; importlib.reload(ra)
import numpy as np, pandas as pd, torch
from functools import lru_cache
from torch.utils.data import DataLoader, Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import f1_score, accuracy_score
from normalizer import normalize

DEV, OUT = "cuda", "/kaggle/working"
R = ra.Renderer(ra.fetch_fonts(f"{OUT}/fonts"))
paths = glob.glob("/kaggle/input/**/*.csv", recursive=True)
pick = lambda k: [p for p in paths if k in os.path.basename(p).lower()][0]
rtr, rte = pd.read_csv(pick("train")), pd.read_csv(pick("test"))
obj = [c for c in rtr.columns if rtr[c].dtype == object]
TEXT = max(obj, key=lambda c: rtr[c].astype(str).str.len().mean())
LABEL = [c for c in rtr.columns if c != TEXT and rtr[c].dropna().nunique() == 2][0]
def prep(d):
    d = d[[TEXT, LABEL]].dropna().copy(); d.columns = ["text", "label"]
    d["text"] = d["text"].astype(str); d["label"] = d["label"].astype(int)
    return d[d.text.str.strip().str.len() > 0].reset_index(drop=True)
train, test = prep(rtr), prep(rte)
HATE, MAXLEN, EFF_BATCH, EPOCHS, SEED = 1, 128, 32, 2, 42

N_ATK = 1500
rng = random.Random(0)
hate_txt = [test.text[i] for i in sorted(rng.sample(test.index[test.label == HATE].tolist(), N_ATK))]
ben_txt  = [test.text[i] for i in sorted(rng.sample(test.index[test.label != HATE].tolist(), N_ATK))]
ATK = {
    "gated_inert_ALL":           [R.attack(t, "ALL", mode="inert", seed=i)[0] for i, t in enumerate(hate_txt)],
    "gated_adaptive_hasant_ALL": [R.attack(t, "ALL", mode="hasant", seed=i)[0] for i, t in enumerate(hate_txt)],
    "gated_both_ALL":            [R.attack(t, "ALL", mode="both", seed=i)[0] for i, t in enumerate(hate_txt)],
}
BEN_ATK = [R.attack(t, "ALL", mode="both", seed=i)[0] for i, t in enumerate(ben_txt)]
DEFENSES = {"none": lambda t: t, "nfc": ra.defend_nfc, "strip_all": ra.defend_strip_all,
            "position_aware": ra.defend_position_aware, "render_equivalence": R.defend_render_equivalence}
DEFENDED = {(a, d): [fn(t) for t in ATK[a]] for a in ATK for d, fn in DEFENSES.items()}
CLEAN_DEF = {d: [fn(t) for t in test.text] for d, fn in DEFENSES.items()}

@lru_cache(maxsize=None)
def norm(t): return normalize(t)

class DS(Dataset):
    """Holds unpadded ids so the collator can pad per batch or to a fixed width."""
    def __init__(self, tk, texts, labels):
        self.ids = tk([norm(t) for t in texts], truncation=True, max_length=MAXLEN)["input_ids"]
        self.y = list(labels)
    def __len__(self): return len(self.y)
    def __getitem__(self, i): return self.ids[i], self.y[i]

def make_collate(tk, dynamic):
    def fn(batch):
        ids = [b[0] for b in batch]; ys = torch.tensor([b[1] for b in batch])
        kw = {"padding": True} if dynamic else {"padding": "max_length", "max_length": MAXLEN}
        enc = tk.pad({"input_ids": ids}, return_tensors="pt", **kw)
        return enc["input_ids"], enc["attention_mask"], ys
    return fn

def make_opt(mdl, lr, exclude_ln):
    if not exclude_ln:
        return torch.optim.AdamW(mdl.parameters(), lr=lr)
    decay, no_decay = [], []
    for n, p in mdl.named_parameters():
        if not p.requires_grad: continue
        flat = n.lower()
        (no_decay if (flat.endswith(".bias") or "layernorm" in flat or "layer_norm" in flat)
         else decay).append(p)
    return torch.optim.AdamW([{"params": decay, "weight_decay": 0.01},
                              {"params": no_decay, "weight_decay": 0.0}], lr=lr)
print("setup done | train", len(train), "| test", len(test))

In [ ]:
# PART A - isolate the cause. 12 steps per configuration, no full training.
MD = "microsoft/mdeberta-v3-base"

def probe(tag, dynamic, exclude_ln, ckpt, lr=1e-5, steps=12, eager=True):
    torch.manual_seed(SEED); np.random.seed(SEED); random.seed(SEED)
    tk = AutoTokenizer.from_pretrained(MD)
    kw = {"num_labels": 2}
    if eager: kw["attn_implementation"] = "eager"
    mdl = AutoModelForSequenceClassification.from_pretrained(MD, **kw).to(DEV)
    if ckpt: mdl.gradient_checkpointing_enable()
    dl = DataLoader(DS(tk, train.text[:2000], train.label[:2000]), batch_size=8,
                    shuffle=True, collate_fn=make_collate(tk, dynamic),
                    generator=torch.Generator().manual_seed(SEED))
    opt = make_opt(mdl, lr, exclude_ln)
    mdl.train()
    bad_grad, bad_step, bad_loss_step = None, None, None
    for st, (i_, m_, y_) in enumerate(dl):
        if st >= steps: break
        i_, m_, y_ = i_.to(DEV), m_.to(DEV), y_.to(DEV)
        opt.zero_grad()
        loss = mdl(input_ids=i_, attention_mask=m_, labels=y_).loss
        if not torch.isfinite(loss):
            bad_loss_step = st; break
        loss.backward()
        for n, p in mdl.named_parameters():
            if p.grad is not None and not torch.isfinite(p.grad).all():
                bad_grad, bad_step = n, st; break
        if bad_grad: break
        torch.nn.utils.clip_grad_norm_(mdl.parameters(), 1.0)
        opt.step()
    del mdl; torch.cuda.empty_cache()
    return {"config": tag, "dynamic_pad": dynamic, "exclude_ln_decay": exclude_ln,
            "grad_ckpt": ckpt, "steps_survived": steps if not (bad_grad or bad_loss_step is not None) else (bad_step if bad_grad else bad_loss_step),
            "first_bad_grad": bad_grad or "", "nonfinite_loss_at": bad_loss_step,
            "ok": bad_grad is None and bad_loss_step is None}

PROBES = [("A as-is", False, False, True), ("B dynamic padding", True, False, True),
          ("C no decay on LayerNorm", False, True, True), ("D no checkpointing", False, False, False),
          ("E all three", True, True, False)]
diag = []
for tag, dyn, exln, ck in PROBES:
    r = probe(tag, dyn, exln, ck)
    diag.append(r); print("  ", r)
diag = pd.DataFrame(diag); diag.to_csv(f"{OUT}/step11_diagnosis.csv", index=False)
print("\n", diag.to_string(index=False))
WINNER = next((r for _, r in diag.iterrows() if r["ok"]), None)
print("\nwinning configuration:", WINNER["config"] if WINNER is not None else "NONE - mDeBERTa cannot train here")

In [ ]:
# PART B and C - train for real, with a NaN-gradient skip so one batch cannot kill a run
def train_full(name, lr, dynamic, exclude_ln, ckpt, micro=8, eager=False):
    accum = max(1, EFF_BATCH // micro)
    torch.manual_seed(SEED); np.random.seed(SEED); random.seed(SEED); torch.cuda.manual_seed_all(SEED)
    tk = AutoTokenizer.from_pretrained(name)
    kw = {"num_labels": 2}
    if eager: kw["attn_implementation"] = "eager"
    mdl = AutoModelForSequenceClassification.from_pretrained(name, **kw).to(DEV)
    if getattr(mdl.config, "pad_token_id", None) is None: mdl.config.pad_token_id = tk.pad_token_id
    if ckpt: mdl.gradient_checkpointing_enable()
    dl = DataLoader(DS(tk, train.text, train.label), batch_size=micro, shuffle=True,
                    collate_fn=make_collate(tk, dynamic),
                    generator=torch.Generator().manual_seed(SEED))
    steps = (len(dl) // accum) * EPOCHS
    opt = make_opt(mdl, lr, exclude_ln)
    sch = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, total_steps=max(1, steps), pct_start=0.1)
    mdl.train(); skipped = done = 0
    for ep in range(EPOCHS):
        opt.zero_grad()
        for st, (i_, m_, y_) in enumerate(dl):
            i_, m_, y_ = i_.to(DEV), m_.to(DEV), y_.to(DEV)
            loss = mdl(input_ids=i_, attention_mask=m_, labels=y_).loss / accum
            if torch.isfinite(loss): loss.backward()
            else: skipped += 1
            if (st + 1) % accum == 0:
                finite = all(torch.isfinite(p.grad).all() for p in mdl.parameters() if p.grad is not None)
                if finite:
                    torch.nn.utils.clip_grad_norm_(mdl.parameters(), 1.0); opt.step()
                else:
                    skipped += 1
                opt.zero_grad()
                if done < steps: sch.step(); done += 1
            if st % 1500 == 0: print(f"    ep{ep} {st}/{len(dl)} loss {loss.item()*accum:.4f} skipped {skipped}")
    mdl.eval(); print(f"    optimizer steps taken {done}, skipped {skipped}")
    return tk, mdl, skipped

@torch.no_grad()
def predict(tk, mdl, texts, bs=16):
    ids = tk([norm(t) for t in texts], truncation=True, max_length=512)["input_ids"]
    order = np.argsort([len(z) for z in ids]); out = np.empty(len(texts), dtype=int); s = 0
    while s < len(order):
        idx = order[s:s+bs]
        try:
            b = tk.pad({"input_ids": [ids[i] for i in idx]}, return_tensors="pt")
            lg = mdl(input_ids=b["input_ids"].to(DEV), attention_mask=b["attention_mask"].to(DEV)).logits
            out[idx] = lg.float().argmax(-1).cpu().numpy(); s += bs
        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache()
            if bs == 1: raise
            bs = max(1, bs // 2); print(f"      inference OOM, batch -> {bs}")
    return out

def evaluate(name, family, tk, mdl, skipped):
    ty = test.label.values
    clean = [{"model": name, "family": family, "defense_on_clean": d,
              "acc": round(accuracy_score(ty, p), 4), "f1": round(f1_score(ty, p), 4)}
             for d in DEFENSES for p in [predict(tk, mdl, CLEAN_DEF[d])]]
    tgt = np.where(predict(tk, mdl, hate_txt) == HATE)[0]
    lad = [{"model": name, "family": family, "tokenizer": type(tk).__name__, "config": c,
            "n_targets": len(tgt), "skipped_steps": skipped,
            "evasion": round(float((predict(tk, mdl, [ATK[c][i] for i in tgt]) != HATE).mean()), 4)}
           for c in ATK]
    dfn = [{"model": name, "attack": a, "defense": d,
            "evasion": round(float((predict(tk, mdl, [DEFENDED[(a, d)][i] for i in tgt]) != HATE).mean()), 4)}
           for a in ATK for d in DEFENSES]
    bp, bpa = predict(tk, mdl, ben_txt), predict(tk, mdl, BEN_ATK)
    ben = {"model": name, "family": family,
           "benign_fpr_clean": round(float((bp == HATE).mean()), 4),
           "benign_fpr_attacked": round(float((bpa == HATE).mean()), 4),
           "benign_flip_rate": round(float((bp != bpa).mean()), 4)}
    for r in lad: print("   ", r)
    print("   ", clean[0], "\n   ", ben)
    return clean, lad, dfn, ben

CL, LA, DE, BE, ST = [], [], [], [], []
JOBS = []
if WINNER is not None:
    JOBS.append((MD, "SentencePiece", 1e-5, bool(WINNER["dynamic_pad"]),
                 bool(WINNER["exclude_ln_decay"]), bool(WINNER["grad_ckpt"]), 8, True))
# BanglaT5 at a learning rate T5 actually needs
JOBS.append(("csebuetnlp/banglat5", "SentencePiece", 1e-4, False, True, True, 8, False))

for name, fam, lr, dyn, exln, ck, micro, eager in JOBS:
    t0 = time.time()
    try:
        tk, mdl, sk = train_full(name, lr, dyn, exln, ck, micro=micro, eager=eager)
        c, l, d, b = evaluate(name, fam, tk, mdl, sk)
        CL += c; LA += l; DE += d; BE.append(b)
        ST.append({"model": name, "ok": True, "lr": lr, "dynamic_pad": dyn,
                   "exclude_ln_decay": exln, "grad_ckpt": ck, "skipped_steps": sk, "error": ""})
        del mdl; torch.cuda.empty_cache()
        print(f"== {name} OK in {(time.time()-t0)/60:.1f} min\n")
    except Exception as ex:
        ST.append({"model": name, "ok": False, "lr": lr, "dynamic_pad": dyn,
                   "exclude_ln_decay": exln, "grad_ckpt": ck, "skipped_steps": None,
                   "error": f"{type(ex).__name__}: {str(ex)[:200]}"})
        print(f"== {name} FAILED: {type(ex).__name__}: {ex}")
        torch.cuda.empty_cache()
    for df, fn in [(CL,"clean_cost"),(LA,"ladder"),(DE,"defense"),(BE,"benign"),(ST,"status")]:
        pd.DataFrame(df).to_csv(f"{OUT}/step11_{fn}.csv", index=False)

In [ ]:
pd.set_option("display.width", 250)
print("="*96); print("STEP 11 SUMMARY"); print("="*96)
print("-- Part A: which difference from standard practice causes the NaN --")
print(diag.to_string(index=False))
print("\n-- Part B/C: training outcome --"); print(pd.DataFrame(ST).to_string(index=False))
if LA: print("\n-- evasion --"); print(pd.DataFrame(LA).to_string(index=False))
if CL: print("\n-- clean accuracy --"); print(pd.DataFrame(CL).query("defense_on_clean=='none'").to_string(index=False))
if DE: print("\n-- defenses --"); print(pd.DataFrame(DE).to_string(index=False))
if BE: print("\n-- benign control --"); print(pd.DataFrame(BE).to_string(index=False))
print("="*96)